# 📓 Notebook 1 — Chuẩn bị dữ liệu (tự crawl)

**Đề tài**: Phân loại rau sạch / rau hỏng bằng Deep Learning

Notebook này thực hiện end-to-end khâu chuẩn bị, **TỰ CRAWL** ảnh từ web (theo yêu cầu đề bài):

1. Setup môi trường + cài deps
2. **Crawl ảnh** từ Google + Bing + Baidu Images theo nhiều keyword
3. Sanity check: thống kê theo keyword, kích thước, file lỗi
4. Tiền xử lý: resize 224×224, loại trùng/lỗi, split train/valid/test 70/15/15
5. Trực quan hoá: ảnh mẫu, phân bố class, augmentation demo

Sau khi chạy xong → tiếp tục với **`02_train_and_evaluate.ipynb`**.

💡 Crawl 10.000 ảnh mất ~30-60 phút. Có thể chạy `--target` nhỏ hơn để test nhanh.

## 1. Thiết lập đường dẫn project

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path('..').resolve()
if Path.cwd().name == 'notebook':
    os.chdir(ROOT)
sys.path.insert(0, str(Path.cwd()))

print('📁 CWD :', Path.cwd())
print('🐍 Py  :', sys.version.split()[0])

## 2. Cài đặt dependencies (lần đầu)

In [ ]:
# Bỏ comment dòng dưới nếu chưa cài
# !pip install -q -r requirements.txt

import tensorflow as tf
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image

print('TF      :', tf.__version__)
print('GPU     :', tf.config.list_physical_devices('GPU') or '(CPU only)')

## 3. Crawl ảnh từ Google + Bing + Baidu

Cấu hình trong `crawler/crawl_images.py`:
- **41 keyword** đa dạng (Anh + Việt) cho fresh và rotten
- 3 search engine song song để tăng đa dạng
- Mỗi keyword lưu thành `<class>/<slug>_<idx>.jpg` để sau biết ảnh thuộc keyword nào

💡 `--target 10000` → mỗi class ~5000 ảnh. Có thể giảm xuống `2000` để test trước.

In [ ]:
# Bỏ comment để chạy crawl. Mất 30-60 phút lần đầu.
# !python crawler/crawl_images.py --target 10000 --engines google,bing,baidu

# Hoặc test nhanh:
# !python crawler/crawl_images.py --target 2000 --engines google,bing

## 4. Kiểm tra dataset đã crawl

In [ ]:
from collections import Counter
RAW = Path('dataset/raw')
assert RAW.exists(), 'Chưa có dataset/raw — chạy cell crawl trước'

stats = {d.name: sum(1 for f in d.iterdir() if f.is_file())
         for d in RAW.iterdir() if d.is_dir()}
print('📦 Số ảnh từng class:')
for k, v in stats.items():
    print(f'   {k:<8s}: {v:,}')
print(f'   {"TỔNG":<8s}: {sum(stats.values()):,}')

In [ ]:
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm
import re

broken, small, sizes, keywords = [], [], [], Counter()
for cls_dir in sorted(d for d in RAW.iterdir() if d.is_dir()):
    for f in tqdm(list(cls_dir.iterdir()), desc=cls_dir.name):
        try:
            with Image.open(f) as im:
                w, h = im.size
                sizes.append((w, h))
                if min(w, h) < 64: small.append(f)
        except (UnidentifiedImageError, OSError):
            broken.append(f)
        m = re.match(r'(.+)_\d+$', f.stem)
        kw = m.group(1) if m else f.stem
        keywords[kw] += 1

print(f'\n✅ Tổng ảnh quét  : {len(sizes):,}')
print(f'❌ Ảnh hỏng       : {len(broken)}')
print(f'⚠️  Ảnh < 64px    : {len(small)}')
print(f'🔤 Số keyword     : {len(keywords)}')
print(f'\n🔤 Top 15 keyword nhiều ảnh nhất:')
for kw, n in keywords.most_common(15):
    print(f'   {kw:<35s}: {n:,}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
bars = axes[0].bar(stats.keys(), stats.values(), color=['#2E7D32', '#C62828'])
for b, v in zip(bars, stats.values()):
    axes[0].text(b.get_x()+b.get_width()/2, v, f'{v:,}', ha='center', va='bottom')
axes[0].set_title('Phân bố số ảnh theo class', fontweight='bold')
axes[0].set_ylabel('Số ảnh')
ws = [w for w, h in sizes]
axes[1].hist(ws, bins=40, color='#1976D2', edgecolor='white')
axes[1].set_title('Phân bố chiều rộng ảnh (px)', fontweight='bold')
axes[1].set_xlabel('Chiều rộng (px)'); axes[1].set_ylabel('Số ảnh')
axes[1].axvline(224, color='red', ls='--', label='Target 224')
axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
import random
random.seed(42)
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for row, cls in enumerate(['fresh', 'rotten']):
    files = random.sample([f for f in (RAW / cls).iterdir() if f.is_file()], 8)
    for col, f in enumerate(files):
        try: axes[row, col].imshow(Image.open(f))
        except Exception: pass
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(cls.upper(), loc='left', fontweight='bold',
                                     fontsize=12, color='#2E7D32' if cls=='fresh' else '#C62828')
plt.suptitle('Ảnh mẫu mỗi class (sau crawl)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 5. Tiền xử lý + split train/valid/test

In [ ]:
!python preprocessing/preprocess.py --src dataset/raw --dst dataset --img-size 224

In [ ]:
split_stats = {}
for split in ('train', 'valid', 'test'):
    sd = Path(f'dataset/{split}')
    if not sd.exists(): continue
    split_stats[split] = {c.name: sum(1 for _ in c.iterdir())
                          for c in sd.iterdir() if c.is_dir()}
df_split = pd.DataFrame(split_stats).fillna(0).astype(int)
print('📊 Phân bố sau split:'); print(df_split)
print(f'\nTỔNG: {df_split.values.sum():,} ảnh')
ax = df_split.plot.bar(figsize=(8, 4),
                       color={'train': '#1976D2', 'valid': '#FFA726', 'test': '#7B1FA2'},
                       edgecolor='white')
ax.set_title('Phân bố train / valid / test', fontweight='bold')
ax.set_ylabel('Số ảnh'); ax.set_xlabel('Class')
ax.set_xticklabels(df_split.index, rotation=0)
for c in ax.containers:
    ax.bar_label(c, label_type='edge', fontsize=8)
plt.tight_layout(); plt.show()

## 6. Demo Augmentation

In [ ]:
from preprocessing.augmentation import build_train_generator
gen = build_train_generator(Path('dataset/train'), img_size=224, batch_size=1)
fig, axes = plt.subplots(2, 4, figsize=(13, 6.5))
for ax in axes.ravel():
    x, _ = next(gen)
    ax.imshow(x[0]); ax.axis('off')
plt.suptitle('8 phiên bản sau augmentation', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## ✅ Hoàn thành chuẩn bị

- `dataset/raw/{fresh,rotten}/` — ảnh gốc tự crawl từ web
- `dataset/processed/` — ảnh đã clean + resize 224×224
- `dataset/{train,valid,test}/` — đã split 70/15/15

👉 **Tiếp theo**: mở `02_train_and_evaluate.ipynb`.